<a href="https://colab.research.google.com/github/harinijk/ImageClassification/blob/main/csci4521hw4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Q1. Identifying Lucky Numbers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import roc_curve, auc, precision_recall_curve
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms.functional import to_pil_image

In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device("cuda" if use_cuda else "cpu")

print("Using GPU:", use_cuda)

In [ ]:
mnist_train = datasets.MNIST(root='./data', train=True, download=True)
mnist_test = datasets.MNIST(root='./data', train=False, download=True)
X_train = mnist_train.data.numpy().reshape(-1, 28*28) / 255.0
X_test = mnist_test.data.numpy().reshape(-1, 28*28) / 255.0
y_train = ((mnist_train.targets.numpy() == 3) |(mnist_train.targets.numpy() == 7) |(mnist_train.targets.numpy() == 8)).astype(int)
y_test = ((mnist_test.targets.numpy() == 3) |(mnist_test.targets.numpy() == 7) |(mnist_test.targets.numpy() == 8)).astype(int)

In [ ]:
log_reg = LogisticRegression(max_iter=1000, solver='lbfgs')
rf = RandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1, random_state=42)

models = [
    ("LogReg", log_reg),
    ("RandomForest", rf)
]

In [ ]:
def evaluate_models(models, X, y):
    for name, model in models:
        print(f"\nEvaluating {name} (5-fold CV)...")

        scores = cross_validate(model, X, y,cv=5,scoring=['accuracy', 'roc_auc', 'average_precision'])
        print("Avg Accuracy:", scores['test_accuracy'].mean())
        print("Avg ROC-AUC:", scores['test_roc_auc'].mean())
        print("Avg PR-AUC:", scores['test_average_precision'].mean())

evaluate_models(models, X_train, y_train)

In [ ]:
for name, model in models:
    print(f"Training {name}...")
    model.fit(X_train, y_train)

probs = {}

for name, model in models:
    probs[name] = model.predict_proba(X_test)[:, 1]

plt.figure(figsize=(8,6))

for name in probs:
    fpr, tpr, _ = roc_curve(y_test, probs[name])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.4f})")

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))

for name in probs:
    precision, recall, _ = precision_recall_curve(y_test, probs[name])
    pr_auc = auc(recall, precision)
    plt.plot(recall, precision, label=f"{name} (AUC={pr_auc:.4f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.show()

In [ ]:

for digit in [3,7,8]:
    print(f"\n Performance for digit {digit} is")

    pred = (y_test == digit) | (y_test == 0)
    y_true = (y_test[pred] == digit).astype(int)

    for name, model in models:
        y_pred = model.predict(X_test[pred])
        acc = accuracy_score(y_true, y_pred)
        print(f"{name} Accuracy: {acc:.4f}")

Question 2

In [ ]:
train_transform = transforms.Compose([transforms.RandomCrop(32, padding=4),transforms.RandomHorizontalFlip(),transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),transforms.ToTensor(),transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
test_transform = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
train_dataset = torchvision.datasets.CIFAR10(root='./data',train=True,download=True,transform=train_transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data',train=False,download=True,transform=test_transform)

In [ ]:
batch_size = 500

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

def train(network, data_loader, loss_function, optimizer):
    network.train()
    avg_loss = 0
    num_batches = 0

    for data, target in data_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = network(data)
        loss = loss_function(output, target)
        loss.backward()
        optimizer.step()
        avg_loss += loss.item()
        num_batches += 1

    return avg_loss / num_batches


def test(network, data_loader, loss_function):
    network.eval()
    total_loss = 0
    num_batches = 0

    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = network(data)
            loss = loss_function(output, target)
            total_loss += loss.item()
            num_batches += 1

    return total_loss / num_batches


def computeAccuracy(network, data_loader):

    correct = 0

    with torch.no_grad():
        for data, target in data_loader:
            data, target = data.to(device), target.to(device)
            output = network(data)
            pred = output.data.max(1, keepdim=True)[1]
            correct += pred.eq(target.view_as(pred)).sum().item()

    return correct / len(data_loader.dataset)

In [ ]:
class Conv_block_batch_norm(nn.Module):
    def __init__(self, channels_in, channels_out):
        super().__init__()
        self.conv = nn.Conv2d(channels_in, channels_out, 3, padding=1)
        self.bn = nn.BatchNorm2d(channels_out)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = F.max_pool2d(x, 2)
        x = F.relu(x)
        return x

In [ ]:
class Conv_residual_block(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, x):
        residual = x
        x = F.relu(self.conv1(x))
        x = self.conv2(x)
        x = x + residual
        x = F.relu(x)
        return x

In [ ]:
class CNN_best(nn.Module):
    def __init__(self, img_w=32, img_h=32, num_classes=10):
        super().__init__()

        self.block1 = Conv_block_batch_norm(3, 32)
        img_w //= 2
        img_h //= 2

        self.res1a = Conv_residual_block(32)
        self.res1b = Conv_residual_block(32)
        self.block2 = Conv_block_batch_norm(32, 64)
        img_w //= 2
        img_h //= 2

        self.res2a = Conv_residual_block(64)
        self.res2b = Conv_residual_block(64)

        self.block3 = Conv_block_batch_norm(64, 128)
        img_w //= 2
        img_h //= 2

        self.res3a = Conv_residual_block(128)
        self.flattened_dim = img_w * img_h * 128
        self.fc1 = nn.Linear(self.flattened_dim, 256)
        self.fc2 = nn.Linear(256, num_classes)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.block1(x)
        x = self.res1a(x)
        x = self.res1b(x)

        x = self.block2(x)
        x = self.res2a(x)
        x = self.res2b(x)

        x = self.block3(x)
        x = self.res3a(x)

        x = x.view(-1, self.flattened_dim)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)

        return x

In [ ]:
model = CNN_best().to(device)
num_epochs = 40
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
# reference: https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.CosineAnnealingLR.html
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
loss_function = nn.CrossEntropyLoss(label_smoothing=0.1)
train_loss_history = []
test_loss_history = []

for epoch in range(num_epochs):
    train_loss = train(model, train_loader, loss_function, optimizer)
    test_loss = test(model, test_loader, loss_function)
    scheduler.step()
    train_loss_history.append(train_loss)
    test_loss_history.append(test_loss)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Test Loss={test_loss:.4f}")


In [ ]:
plt.plot(train_loss_history, label='Train Loss')
plt.plot(test_loss_history, label='Test Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training vs Test Loss")
plt.show()

In [ ]:
train_acc = computeAccuracy(model, train_loader)
test_acc = computeAccuracy(model, test_loader)

print("Train Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Question 3

In [ ]:
torch.save(model.state_dict(), "best_model.pt")

In [ ]:
model_loaded = CNN_best().to(device)
model_loaded.load_state_dict(torch.load("best_model.pt"))
model_loaded.eval()
train_acc_saved = computeAccuracy(model_loaded, train_loader)
print("saved model train accuracy:", train_acc_saved)
saved_test_acc = computeAccuracy(model_loaded, test_loader)
print("saved model test accuracy:", saved_test_acc)

Question 4

In [ ]:
model_loaded.eval()
test_acc = computeAccuracy(model_loaded, test_loader)
print("Test Accuracy:", test_acc)

all_labels = []
all_preds = []

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        output = model_loaded(data)
        pred = output.argmax(dim=1)

        all_labels.extend(target.cpu().numpy())
        all_preds.extend(pred.cpu().numpy())


conf_mat = confusion_matrix(all_labels, all_preds)
print(conf_mat)

In [ ]:
conf_mat_norm = confusion_matrix(all_labels, all_preds, normalize="true")

plt.figure(figsize=(8,6))
sns.heatmap(conf_mat_norm, annot=True, fmt=".2f", cmap="gnuplot")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
class_names = train_dataset.classes
confidence = []
images = []

for i in range(10):
    confidence_value = 0.0
    img_value = None

    confidence.append(confidence_value)
    images.append(img_value)
model_loaded.eval()
with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        output = model_loaded(data)
        probs = F.softmax(output, dim=1)
        pred = output.argmax(dim=1)

        for i in range(len(target)):
            true_class = target[i].item()

            if pred[i].item() == true_class:
                confidence_new = probs[i][true_class].item()

                if confidence_new > confidence[true_class]:
                    confidence[true_class] = confidence_new
                    images[true_class] = data[i]

In [ ]:
plt.figure(figsize=(12,6))

for i in range(10):
    plt.subplot(2,5,i+1)

    if images[i] is not None:
        img = images[i].cpu().permute(1,2,0)
        plt.imshow(img)
        plt.title(class_names[i])

plt.suptitle("Most Confident Images")
plt.show()

In [ ]:
errors = {}
model_loaded.eval()

with torch.no_grad():
    for data, target in test_loader:
        data = data.to(device)
        output = model_loaded(data)
        pred = output.argmax(dim=1)

        for t, p in zip(target, pred):
            if t.item() != p.item():
                key = (t.item(), p.item())
                errors[key] = errors.get(key, 0) + 1


error_list = []
for key in errors:
    error_list.append((key, errors[key]))

for i in range(len(error_list)):
    for j in range(i + 1, len(error_list)):
        if error_list[j][1] > error_list[i][1]:
            temp = error_list[i]
            error_list[i] = error_list[j]
            error_list[j] = temp

top_errors = []
for i in range(5):
    if i < len(error_list):
        top_errors.append(error_list[i])


In [ ]:
for error in top_errors:
    true_class, pred_class = error[0]

    print(f"{class_names[true_class]} but predict {class_names[pred_class]}")

    plt.figure(figsize=(8,2))
    count = 0

    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device)

            output = model_loaded(data)
            pred = output.argmax(dim=1)

            for i in range(len(target)):
                t = target[i].item()
                p = pred[i].item()

                if t == true_class and p == pred_class:

                    plt.subplot(1,4,count+1)

                    img = data[i].cpu().permute(1,2,0)
                    img = img * 0.5 + 0.5

                    plt.imshow(img)
                    count += 1
                    if count == 4:
                        break

            if count == 4:
                break

    plt.show()